In [2]:
import numpy as np
import torch
import time
from transformers import Trainer, TrainingArguments

from evaluation.metrics_trainer_callback import SaveMetricsCallback
from evaluation.metrics import compute_ece, compute_B_std, compute_B_mean
from data_loading.get_datasets import get_glue_dataset
from models.get_roberta import get_hypernet_on_last_layer_roberta

AttributeError: partially initialized module 'torch' has no attribute 'distributed' (most likely due to a circular import)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

glue_dataset_name = "cola"
model_name = "roberta-base"
lora_r = 1
lora_alpha = 16
hypernet_hidden_dim = 16

print(f"Hypernet on: {glue_dataset_name}")

Hypernet on: sst2


In [ ]:
model, tokenizer, hypernet = get_hypernet_on_last_layer_roberta(model_name=model_name, lora_r=lora_r, lora_alpha=lora_alpha, hypernet_hidden_dim=hypernet_hidden_dim)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
encoded_dataset, metric = get_glue_dataset(glue_dataset_name, tokenizer, truncation=True, max_length=512)

In [ ]:
def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()

        predictions = np.argmax(probs, axis=-1)
        results = metric.compute(predictions=predictions, references=labels)

        results["ece"] = compute_ece(probs, labels)
        results["hyper_B_std"] = compute_B_std(hypernet, device=device)
        results["hyper_B_mean"] = compute_B_mean(hypernet, device=device)

        return results

In [ ]:
training_args = TrainingArguments(
    output_dir=f"./outputs/hypernet_{glue_dataset_name}",
    eval_strategy="steps",
    eval_steps=5,
    save_strategy="steps",
    save_steps=1000000000,
    learning_rate=4e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=32,
    num_train_epochs=20, # 80
    metric_for_best_model="matthews_correlation",
    dataloader_num_workers=4,
    warmup_ratio=0.06,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    weight_decay=0.1,
    disable_tqdm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[SaveMetricsCallback(f"./results", f"hypernet_{glue_dataset_name}_{str(int(time.time()))}.csv")]
)

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
def grad_hook(grad):
    print(f"[BACKWARD] grad shape: {grad.shape}, mean: {grad.mean().item():.8f} HYPERNET")
    return grad

hypernet.fc1.weight.register_hook(grad_hook)

model.roberta.encoder.layer[-1]

RobertaLayer(
  (attention): RobertaAttention(
    (self): RobertaSdpaSelfAttention(
      (query): lora.Linear(
        (base_layer): Linear(in_features=768, out_features=768, bias=True)
        (lora_dropout): ModuleDict(
          (default): Identity()
        )
        (lora_A): ModuleDict(
          (default): DynamicLoRALayer(
            (hypernet): LoRAHyperNet(
              (fc1): Linear(in_features=776, out_features=16, bias=True)
              (fc2): Linear(in_features=16, out_features=768, bias=True)
              (embedding): Embedding(2, 8)
            )
          )
        )
        (lora_B): ModuleDict(
          (default): Identity()
        )
        (lora_embedding_A): ParameterDict()
        (lora_embedding_B): ParameterDict()
        (lora_magnitude_vector): ModuleDict()
      )
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): lora.Linear(
        (base_layer): Linear(in_features=768, out_features=768, bias=True)
        (lora_dropo

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,Ece,Hyper B Std,Hyper B Mean
5,No log,0.703893,0.504587,0.068092,0.158447,0.002622


KeyError: "The `metric_for_best_model` training argument is set to 'eval_matthews_correlation', which is not found in the evaluation metrics. The available evaluation metrics are: ['eval_loss', 'eval_accuracy', 'eval_ece', 'eval_hyper_B_std', 'eval_hyper_B_mean']. Consider changing the `metric_for_best_model` via the TrainingArguments."